# Dataset 7: Friday-PortScan EDA
This notebook analyzes reconnaissance port scanning and investigates the 25.26% duplicate rate.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv", encoding='latin1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

cols = list(df.columns)
if cols.count('Fwd Header Length') > 1:
    first_idx = cols.index('Fwd Header Length')
    second_idx = cols.index('Fwd Header Length', first_idx + 1)
    cols[second_idx] = 'Fwd Header Length.1'
    df.columns = cols

print(f"Loaded Friday PortScan: {df.shape[0]:,} rows")
print(f"Class breakdown:\n{df['Label'].value_counts()}")


## 1. Single-Packet Probes & High Duplicate Rate Investigation

In [ ]:
for c in [c for c in df.columns if c != 'Label']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

scan_df = df[df['Label'] == 'PortScan']
print(f"Total Duplicates in PortScan Class: {scan_df.duplicated().sum():,} / {len(scan_df):,} ({round(scan_df.duplicated().mean()*100, 2)}%)")
print(f"1-Packet Probe Flows: {(scan_df['Total Fwd Packets'] == 1).sum():,} ({round((scan_df['Total Fwd Packets'] == 1).mean()*100, 2)}%)")
print(f"Unique Ports Probed by PortScan: {scan_df['Destination Port'].nunique():,}")

plt.figure(figsize=(10, 4.5))
sns.histplot(scan_df['Destination Port'], bins=60, color='#e41a1c', label='PortScan')
sns.histplot(df[df['Label'] == 'BENIGN']['Destination Port'], bins=60, color='#2b5c8f', alpha=0.3, label='BENIGN')
plt.legend()
plt.title("Friday Afternoon: Destination Port Spectrum")
plt.show()


## 2. Key Findings & Preprocessing Recommendations
- 42.86% of PortScan attack records are exact duplicates due to identical 1-packet TCP probe vectors across 1,000 ports.
- Deduplication is mandatory prior to train/test partitioning to prevent data leakage.
